# Disaster video analysis — **GPU + long-gap ReID + local VLM**

**No paid API.** This notebook runs in Google Colab with a GPU; model checkpoints download from public repositories. It analyzes videos locally inside your Colab session (not on your home PC). Colab's free GPU is subject to availability and usage quotas.

**How the counts are produced**

1. **YOLO pose + BoT-SORT** detects/tracks people throughout the video.
2. **DINOv2 appearance embeddings** and clothing comparison match *tracklets* after ID loss. This is a second, video-wide matching pass; it does **not** simply count YOLO tracker IDs.
3. **Qwen3-VL-2B-Instruct**, loaded on the GPU in 4-bit, reviews timestamped scene sheets and individual-person crops, describing visible distress and possible assistance cues. The VLM **never controls the number of unique people** or invents unseen individuals.
4. Reports include **estimated unique people**, **visibly concerning people**, **unseen after the reported explosion**, and the **union requiring a safety check** (each individual counted once).
5. Reviewable **automatic merges**, ambiguous identity pairs, images, annotated video, CSV files, and a JSON report are exported.

**Critical distinction:** a person hidden by smoke or who is no longer shown is *not* proven injured, missing, or in need of medical treatment. The notebook can count them as **welfare-check priority (status unknown)**, **not confirmed casualties**. A person who reappears is counted only once *when reliably matched*. Same-looking clothing and camera outages can fool tracking; all totals are estimates. This system must **not** be used as the sole basis for emergency triage or rescue decisions.

Recommended: **Runtime → Change runtime type → T4 GPU** (or better). Run the cells in order. Set the explosion time for your clip in Step 2 (for your earlier shop video, approximately `12.0`).

## 1. Install dependencies and verify GPU
No OpenAI, Gemini, or other paid service is called. Models download once from Hugging Face/Ultralytics into Colab's runtime.

In [ ]:
%pip -q install -U ultralytics transformers accelerate bitsandbytes opencv-python-headless pandas Pillow PyYAML safetensors

In [ ]:
import torch, os
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. In Colab choose Runtime > Change runtime type > T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print('Torch:', torch.__version__)
DEVICE = 0

## 2. Upload video and configure thresholds
- For the earlier explosion clip, use `EVENT_TIME_SECONDS = 12.0`. **Change this for a different video.** If no verified event time is available, set `None`, and the disappearance-based category will be disabled rather than inventing an event.
- `AUTO_MERGE_THRESHOLD` is intentionally conservative. Look-alike clothing can still lead to incorrect merges, and changing it changes counts. Review `possible_duplicate_pairs.csv`.
- For a quick run use `MAX_FRAMES = 300`, but **do not report that as the whole video**.
- Person VLM review takes longer for larger crowds. Raise `MAX_PERSON_VLM_REVIEWS` as needed; it does not process more than that number.

In [ ]:
from pathlib import Path
import math, json, re, gc, shutil, subprocess, itertools, warnings
from collections import defaultdict
from itertools import combinations
import numpy as np, pandas as pd, cv2, yaml
from PIL import Image as PILImage, ImageDraw
from IPython.display import display, Image as NotebookImage, Video as NotebookVideo
from ultralytics import YOLO
import ultralytics

VIDEO_PATH = '/content/i-uKPQCdJdzoc_rl.mp4'                       # Empty opens the Colab upload dialog
OUTPUT_DIR = Path('/content/disaster_ai_outputs')
EVENT_TIME_SECONDS = 12.0            # Earlier clip: about 12s. Change or use None.
EVENT_BEFORE_WINDOW_SECONDS = 4.0
EVENT_AFTER_GRACE_SECONDS = 1.5
POSE_MODEL = 'yolo26n-pose.pt'
MIN_DETECTION_CONFIDENCE = 0.22
TRACK_BUFFER_FRAMES = 120
MIN_TRACK_OBSERVATIONS = 4
MAX_FRAMES = None                    # None means the whole video
DETECTION_IMG_SIZE = 640
MIN_BODY_CROP_WH = (18, 35)
CROP_SAMPLE_INTERVAL_SECONDS = 0.70
MAX_CROPS_PER_TRACK = 8            # Per tracklet, reservoir-sampled over its WHOLE duration
REID_MODEL = 'facebook/dinov2-small'
AUTO_MERGE_THRESHOLD = 0.92          # Tune on labeled videos; not universal
AUTO_MERGE_MARGIN = 0.055            # Difference from next plausible match
AUTO_MERGE_MAX_GAP_SECONDS = 12.0
REVIEW_MAX_GAP_SECONDS = 45.0
REVIEW_MIN_SCORE = 0.80
HORIZONTAL_POSTURE_SECONDS = 1.7
# VLM defaults suitable for typical 16 GB Colab T4
RUN_VLM = True
VLM_MODEL = 'Qwen/Qwen3-VL-2B-Instruct' # Optional: Qwen/Qwen3-VL-4B-Instruct
MAX_SCENE_SHEETS = 6
MAX_PERSON_VLM_REVIEWS = 20
VLM_MAX_NEW_TOKENS = 220

OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
if not VIDEO_PATH:
    from google.colab import files
    print('Upload one MP4/MOV video:')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No video selected')
    VIDEO_PATH = next(iter(uploaded))
if not Path(VIDEO_PATH).is_file():
    raise FileNotFoundError(VIDEO_PATH)
print('Video:', VIDEO_PATH)

## 3. Tracker with GPU pose detection
Track IDs are **temporary tracklets**, *not* the unique-person count. We sample actual cropped appearances from each tracklet for later matching. **Co-visible** tracks may never be merged: they represent different people in the same frame.

In [ ]:
base = Path(ultralytics.__file__).resolve().parent / 'cfg' / 'trackers' / 'botsort.yaml'
tracker_cfg = yaml.safe_load(base.read_text())
tracker_cfg.update({'with_reid': True, 'model': 'auto', 'track_buffer': TRACK_BUFFER_FRAMES,
                    'appearance_thresh': 0.83})
tracker_yaml = OUTPUT_DIR / 'botsort_reid.yaml'
tracker_yaml.write_text(yaml.safe_dump(tracker_cfg))

def is_horizontal(box, kpts):
    x1,y1,x2,y2 = map(float, box)
    w,h = max(1.,x2-x1),max(1.,y2-y1)
    wide = (w / h) >= 1.30
    if kpts is None or len(kpts) < 13:
        return wide
    kp=np.asarray(kpts)
    if kp.shape[1] < 3 or not all(kp[k,2] >= 0.4 for k in (5,6,11,12)):
        return wide
    shoulder=(kp[5,:2]+kp[6,:2])/2
    hip=(kp[11,:2]+kp[12,:2])/2
    sideways=abs(shoulder[0]-hip[0]) >= 1.45*max(1., abs(shoulder[1]-hip[1]))
    return bool(wide or sideways)

def crop_person(frame,box):
    H,W=frame.shape[:2]
    x1,y1,x2,y2=[int(v) for v in box]
    x1=max(0,x1);y1=max(0,y1);x2=min(W,x2);y2=min(H,y2)
    if x2-x1 < MIN_BODY_CROP_WH[0] or y2-y1 < MIN_BODY_CROP_WH[1]:
        return None
    return cv2.cvtColor(frame[y1:y2,x1:x2],cv2.COLOR_BGR2RGB)

def color_descriptor(rgb):
    h,w=rgb.shape[:2]
    central=rgb[int(h*.18):int(h*.82),int(w*.20):int(w*.80)]
    if central.size == 0: return None
    hsv=cv2.cvtColor(central,cv2.COLOR_RGB2HSV)
    hist=cv2.calcHist([hsv],[0,1],None,[16,8],[0,180,0,256]).reshape(-1)
    return hist / (np.linalg.norm(hist)+1e-9)

cap=cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened(): raise RuntimeError('Cannot open uploaded video')
fps=float(cap.get(cv2.CAP_PROP_FPS))
if not (1 <= fps <= 240):
    warnings.warn('No reliable FPS; using 30')
    fps=30.0
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
if W <= 0 or H <= 0: raise RuntimeError('Video dimensions missing')
model=YOLO(POSE_MODEL)
tracks={}; simultaneous=set(); frames=[]; frame_no=0
last_cropped=defaultdict(lambda:-1e8)
rng=np.random.default_rng(2026)
raw_video=str(OUTPUT_DIR/'raw_tracker_ids.mp4')
writer=cv2.VideoWriter(raw_video,cv2.VideoWriter_fourcc(*'mp4v'),fps,(W,H))
if not writer.isOpened(): raise RuntimeError('Cannot create annotated video')
try:
    while True:
        ok,frm=cap.read()
        if not ok or (MAX_FRAMES is not None and frame_no >= MAX_FRAMES): break
        t=frame_no/fps
        r=model.track(frm,persist=True,tracker=str(tracker_yaml),conf=MIN_DETECTION_CONFIDENCE,
                      imgsz=DETECTION_IMG_SIZE,device=DEVICE,half=True,verbose=False)[0]
        observations=[]
        if r.boxes is not None and r.boxes.id is not None:
            tids=r.boxes.id.int().cpu().tolist()
            boxes=r.boxes.xyxy.cpu().numpy()
            confs=r.boxes.conf.cpu().numpy()
            kp=(r.keypoints.data.cpu().numpy() if r.keypoints is not None else None)
            for j,(tid,box,cf) in enumerate(zip(tids,boxes,confs)):
                tid=int(tid)
                if tid not in tracks:
                    tracks[tid]={'first':t,'last':t,'first_frame':frame_no,'last_frame':frame_no,
                                 'n':0,'start_box':box.copy(),'end_box':box.copy(),'crops':[],
                                 'crop_count':0,
                                 'colors':[],'pose_start':None,'posture_flag':False,'pose_times':[],
                                 'saw_before':False,'saw_after':False,'max_conf':0.0}
                tr=tracks[tid]
                if tr['n'] and frame_no > tr['last_frame']+1:
                    tr['pose_start']=None
                posture=is_horizontal(box,kp[j] if kp is not None and j<len(kp) else None)
                if posture:
                    if tr['pose_start'] is None: tr['pose_start']=t
                    if t-tr['pose_start'] >= HORIZONTAL_POSTURE_SECONDS:
                        tr['posture_flag']=True
                        if len(tr['pose_times'])<5: tr['pose_times'].append(round(t,2))
                else: tr['pose_start']=None
                if EVENT_TIME_SECONDS is not None:
                    if EVENT_TIME_SECONDS-EVENT_BEFORE_WINDOW_SECONDS <= t < EVENT_TIME_SECONDS:
                        tr['saw_before']=True
                    if t >= EVENT_TIME_SECONDS + EVENT_AFTER_GRACE_SECONDS:
                        tr['saw_after']=True
                if t-last_cropped[tid] >= CROP_SAMPLE_INTERVAL_SECONDS:
                    crop=crop_person(frm,box)
                    if crop is not None:
                        tr['crop_count']+=1
                        candidate=(t,PILImage.fromarray(crop))
                        color=color_descriptor(crop)
                        if len(tr['crops'])<MAX_CROPS_PER_TRACK:
                            tr['crops'].append(candidate)
                            tr['colors'].append(color)
                        else:
                            # Reservoir sampling spans the full tracklet, not only its first seconds.
                            slot=int(rng.integers(0,tr['crop_count']))
                            if slot<MAX_CROPS_PER_TRACK:
                                tr['crops'][slot]=candidate
                                tr['colors'][slot]=color
                        last_cropped[tid]=t
                tr['n']+=1; tr['last']=t;tr['last_frame']=frame_no
                tr['end_box']=box.copy();tr['max_conf']=max(tr['max_conf'],float(cf))
                observations.append({'tid':tid,'box':np.asarray(box).tolist(),'posture':bool(posture)})
                x1,y1,x2,y2=map(int,box)
                cv2.rectangle(frm,(x1,y1),(x2,y2),(0,180,255) if posture else (0,230,30),2)
                cv2.putText(frm,f'T{tid}',(max(0,x1),max(20,y1-7)),cv2.FONT_HERSHEY_SIMPLEX,.6,(255,255,255),2)
        ids=sorted({o['tid'] for o in observations})
        simultaneous.update(combinations(ids,2))
        frames.append({'index':frame_no,'sec':t,'obs':observations})
        cv2.putText(frm,f'Visible {len(ids)}   {t:.1f}s',(14,32),cv2.FONT_HERSHEY_SIMPLEX,.8,(255,255,255),3)
        writer.write(frm)
        frame_no+=1
        if frame_no % 150 == 0: print('Analyzed',frame_no,'frames;',round(t,1),'seconds')
finally:
    cap.release();writer.release()
if frame_no==0: raise RuntimeError('No readable frames')
duration=(frame_no-1)/fps
valid_ids={k for k,v in tracks.items() if v['n'] >= MIN_TRACK_OBSERVATIONS}
print(f'{frame_no} frames, {duration:.1f}s, {len(valid_ids)} eligible raw track IDs')
pd.DataFrame([{'raw_id':i,'start_s':v['first'],'end_s':v['last'],'observations':v['n']}
              for i,v in sorted(tracks.items())]).to_csv(OUTPUT_DIR/'raw_tracks.csv',index=False)
# Free the YOLO weights before the next GPU models.
del model,r
if torch.cuda.is_available(): torch.cuda.empty_cache()
gc.collect()

## 4. Extract strong appearance embeddings (DINOv2 on GPU)
This is the **second pass** that fixes the tracker-ID issue: we compare representative person crops from **different track IDs**, including those separated by the explosion. DINOv2 is a general visual encoder, **not a specialist person-ReID model**, so matches remain fallible.

In [ ]:
from transformers import AutoImageProcessor, AutoModel
img_processor=AutoImageProcessor.from_pretrained(REID_MODEL)
embed_model=AutoModel.from_pretrained(REID_MODEL).to('cuda').eval()

for tid in sorted(valid_ids):
    tr=tracks[tid]
    embs=[]
    for start in range(0,len(tr['crops']),4):
        batch=[im for _,im in tr['crops'][start:start+4]]
        inp=img_processor(images=batch,return_tensors='pt').to('cuda')
        with torch.inference_mode():
            res=embed_model(**inp).last_hidden_state[:,0]
            res=torch.nn.functional.normalize(res.float(),p=2,dim=-1)
        embs.extend(res.cpu().numpy())
    tr['embeddings']=np.array(embs,dtype=np.float32)
    cols=[a for a in tr['colors'] if a is not None]
    tr['color_mean']=(np.mean(cols,axis=0) if cols else None)
    if tr['color_mean'] is not None:
        tr['color_mean'] /= np.linalg.norm(tr['color_mean'])+1e-9

del embed_model,img_processor
if torch.cuda.is_available():torch.cuda.empty_cache()
gc.collect()
print('Appearance embeddings ready for',sum(len(tracks[x]['embeddings'])>0 for x in valid_ids),'tracklets')

## 5. Conservative automatic reconnection of lost tracks
Auto-merging requires: no overlap in time (and never simultaneously visible), strong **multi-crop appearance** agreement, acceptable time gap, plausible position for short gaps, and a unique match margin. **Uncertain matches are left for manual review**. It is not possible to guarantee perfect person counts after strong smoke/occlusion.

In [ ]:
def track_pair(a,b):
    A,B=tracks[a],tracks[b]
    if A['last'] < B['first']:
        earlier,later=A,B
    elif B['last'] < A['first']:
        earlier,later=B,A
    else:
        return None  # temporal tracklet overlap: cannot safely be one person
    gap=later['first']-earlier['last']
    if gap>REVIEW_MAX_GAP_SECONDS or (min(a,b),max(a,b)) in simultaneous:
        return None
    E,F=A['embeddings'],B['embeddings']
    if not len(E) or not len(F):return None
    cos=E@F.T
    # Robust to differing angles / partial crops. Both directions must match.
    visual=float((np.median(cos.max(axis=1))+np.median(cos.max(axis=0)))/2)
    color=float(np.dot(A['color_mean'],B['color_mean'])) if A['color_mean'] is not None and B['color_mean'] is not None else 0.
    # Keep hist weak: a similar shirt does not prove identity.
    score=.86*visual+.14*max(0.,color)
    prev=earlier['end_box']; nxt=later['start_box']
    p=np.array([(prev[0]+prev[2])/2,(prev[1]+prev[3])/2])
    q=np.array([(nxt[0]+nxt[2])/2,(nxt[1]+nxt[3])/2])
    relative_distance=float(np.linalg.norm((p-q)/np.array([W,H])))
    # For very short gaps on fixed-camera videos, prevent teleportation.
    spatial_ok=(gap>3.0 or relative_distance <= .23+.03*gap)
    return {'id_a':a,'id_b':b,'score':round(score,5),'dino_cosine':round(visual,5),
            'clothes_sim':round(color,5),'gap_s':round(gap,3),
            'position_change':round(relative_distance,4),'spatial_ok':spatial_ok}

pairs=[]
for a,b in combinations(sorted(valid_ids),2):
    info=track_pair(a,b)
    if info is not None and info['score'] >= REVIEW_MIN_SCORE:
        pairs.append(info)
candidates=pd.DataFrame(pairs,columns=['id_a','id_b','score','dino_cosine','clothes_sim',
                                      'gap_s','position_change','spatial_ok'])
if not candidates.empty:
    candidates=candidates.sort_values(['score','gap_s'],ascending=[False,True]).reset_index(drop=True)

class SafeIdentityGroups:
    def __init__(self,ids,co_visible,blocked=None):
        self.blocked=set(blocked or [])
        self.parent={x:x for x in ids}
        self.members={x:{x} for x in ids}
        self.co_visible=co_visible
    def root(self,x):
        while self.parent[x]!=x:
            self.parent[x]=self.parent[self.parent[x]]
            x=self.parent[x]
        return x
    def union(self,a,b):
        ra,rb=self.root(a),self.root(b)
        if ra==rb: return True
        A,B=self.members[ra],self.members[rb]
        if any(tuple(sorted((x,y))) in (self.co_visible | self.blocked) for x in A for y in B): return False
        # Prevent transitive merging of temporally overlapping raw tracklets.
        if any(not (tracks[x]['last']<tracks[y]['first'] or tracks[y]['last']<tracks[x]['first'])
               for x in A for y in B): return False
        if len(A)<len(B):ra,rb=rb,ra
        self.parent[rb]=ra
        self.members[ra]|=self.members[rb]
        del self.members[rb]
        return True
    def groups(self):return sorted([sorted(v) for v in self.members.values()],key=lambda a:min(a))

identity=SafeIdentityGroups(valid_ids,simultaneous)
# A candidate must stand out against competing choices for BOTH IDs.
def second_best_margin(row):
    s=float(row.score); a=int(row.id_a);b=int(row.id_b)
    competitors=candidates[(candidates.id_a.isin([a,b])|candidates.id_b.isin([a,b])) &
                           ~((candidates.id_a==a)&(candidates.id_b==b))]
    return s-float(competitors.score.max()) if len(competitors) else 1.

auto_rows=[]
if len(candidates):
    for item in candidates.itertuples(index=False):
        margin=second_best_margin(item)
        auto_ok=(item.score>=AUTO_MERGE_THRESHOLD and item.gap_s<=AUTO_MERGE_MAX_GAP_SECONDS
                 and item.spatial_ok and margin>=AUTO_MERGE_MARGIN)
        merged=False
        if auto_ok:
            merged=identity.union(int(item.id_a),int(item.id_b))
        auto_rows.append({**item._asdict(),'separation_margin':round(margin,4),
                          'auto_merged':bool(merged)})
match_table=pd.DataFrame(auto_rows,columns=list(candidates.columns)+['separation_margin','auto_merged'])
match_table.to_csv(OUTPUT_DIR/'possible_duplicate_pairs.csv',index=False)
print('Raw tracker IDs:',len(valid_ids),'| Auto merged pairs:',int(match_table.auto_merged.sum()) if len(match_table) else 0,
      '| Estimated people:',len(identity.groups()))
display(match_table.head(25))

# Save a visual side-by-side review sheet for ambiguous pairs.
def identity_crop(tid):
    return tracks[tid]['crops'][0][1] if tracks[tid]['crops'] else None
interesting=match_table.head(15)
previews=[]
if not interesting.empty:
    for row in interesting.itertuples():
        im_a,im_b=identity_crop(int(row.id_a)),identity_crop(int(row.id_b))
        if im_a is None or im_b is None:continue
        tile=PILImage.new('RGB',(350,235),'white');d=ImageDraw.Draw(tile)
        tile.paste(im_a.resize((150,190)),(10,30));tile.paste(im_b.resize((150,190)),(190,30))
        d.text((10,6),f'T{row.id_a}  vs  T{row.id_b}  score {row.score:.3f}  merged={row.auto_merged}',fill='black')
        previews.append(tile)
if previews:
    preview=PILImage.new('RGB',(350,235*len(previews)),'white')
    for i,im in enumerate(previews):preview.paste(im,(0,235*i))
    preview.save(OUTPUT_DIR/'identity_review.jpg')
    display(NotebookImage(filename=str(OUTPUT_DIR/'identity_review.jpg')))

## 6. Correct any identity mistakes (optional but recommended)
The automatic reconnection above runs without requiring manual merges. To correct mistakes, edit below and rerun this cell and later cells. `EXCLUDE_RAW_IDS` removes false detections. `FORCE_DIFFERENT` blocks a proposed automatic match; `FORCE_SAME` merges tracks if non-overlapping and not co-visible. These lists use **raw tracker IDs** (T numbers).

In [ ]:
EXCLUDE_RAW_IDS = []             # e.g. [42] if a tracker detection is NOT a person
FORCE_DIFFERENT = []             # e.g. [(3,17)] if a wrong automatic merge occurred
FORCE_SAME = [(2,4)]                  # e.g. [(3,17)] if you verified same person
MANUAL_VISIBLE_HELP_RAW_IDS = [] # e.g. [3] if you actually SEE an assistance cue
MANUAL_DISMISS_HELP_RAW_IDS = [] # e.g. [10] if a model alert is clearly wrong

blocked={tuple(sorted((int(a),int(b)))) for a,b in FORCE_DIFFERENT}
active_ids=valid_ids-set(EXCLUDE_RAW_IDS)
if set(EXCLUDE_RAW_IDS)-valid_ids: raise ValueError('An excluded raw ID is unknown')
identity=SafeIdentityGroups(active_ids,simultaneous,blocked)
for row in auto_rows:
    if (row['auto_merged'] and row['id_a'] in active_ids and row['id_b'] in active_ids
        and tuple(sorted((int(row['id_a']),int(row['id_b'])))) not in blocked):
        identity.union(int(row['id_a']),int(row['id_b']))
for a,b in FORCE_SAME:
    if a not in active_ids or b not in active_ids:raise ValueError('Unknown or excluded ID in FORCE_SAME')
    if not identity.union(a,b):
        raise ValueError(f'Cannot safely merge tracks T{a}/T{b}: simultaneous or overlapping')
groups=identity.groups()
raw_to_person={tid:f'P{i:03d}' for i,g in enumerate(groups,1) for tid in g}
person_to_raw={f'P{i:03d}':g for i,g in enumerate(groups,1)}

for key in ['MANUAL_VISIBLE_HELP_RAW_IDS','MANUAL_DISMISS_HELP_RAW_IDS']:
    unknown=set(globals()[key])-active_ids
    if unknown: raise ValueError(f'Unknown IDs in {key}: {unknown}')
print('Estimated unique people (after automatic + optional manual corrections):',len(groups))
print('Identity groups:',person_to_raw)

## 7. Make timestamped frames and person crops for the VLM
The VLM sees **actual visual evidence**, not only numeric tracker outputs. Canonical IDs are drawn on frames after ID merging. The VLM can describe the event and visible distress but cannot independently recover invisible people.

In [ ]:
# Choose frames across the entire analyzed video and densely around the marked event.
n_total=min(4*MAX_SCENE_SHEETS, max(4,round(duration/2)))
sample_times=list(np.linspace(0,duration,n_total))
if EVENT_TIME_SECONDS is not None and EVENT_TIME_SECONDS<=duration:
    sample_times += [EVENT_TIME_SECONDS+d for d in [-2,-1,-.25,.25,1,2,4,7]]
sample_ids=sorted({min(frame_no-1,max(0,round(t*fps))) for t in sample_times})
# Keep event frames even on long footage. Limit only background uniform samples.
if len(sample_ids)>4*MAX_SCENE_SHEETS:
    event_ids={min(frame_no-1,max(0,round((EVENT_TIME_SECONDS+d)*fps)))
               for d in [-2,-1,-.25,.25,1,2,4,7]} if EVENT_TIME_SECONDS is not None else set()
    event_ids={i for i in event_ids if i>=0}
    others=[i for i in sample_ids if i not in event_ids]
    limit=max(0,4*MAX_SCENE_SHEETS-len(event_ids))
    keep=[others[int(j)] for j in np.linspace(0,len(others)-1,limit)] if others and limit else []
    sample_ids=sorted(event_ids.union(keep))

cap=cv2.VideoCapture(VIDEO_PATH)
views=[]
try:
    for fi in sample_ids:
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(fi));ok,frame=cap.read()
        if not ok:continue
        # Scale the frame for readable panels, then draw canonical IDs in original coordinates.
        for det in frames[fi]['obs']:
            tid=det['tid']
            if tid not in raw_to_person:continue
            x1,y1,x2,y2=map(int,det['box'])
            cv2.rectangle(frame,(x1,y1),(x2,y2),(38,235,70),3)
            cv2.putText(frame,raw_to_person[tid],(max(0,x1),max(20,y1-8)),
                        cv2.FONT_HERSHEY_SIMPLEX,.8,(255,255,255),3)
        im=PILImage.fromarray(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
        panel=PILImage.new('RGB',(480,300),'#121212')
        im.thumbnail((480,270))
        panel.paste(im,((480-im.width)//2,25+(270-im.height)//2))
        ImageDraw.Draw(panel).text((8,4),f't = {fi/fps:.2f}s',fill='white')
        views.append(panel)
finally:cap.release()

scene_sheets=[]
for k in range(0,len(views),4):
    sheet=PILImage.new('RGB',(960,600),'white')
    for j,im in enumerate(views[k:k+4]):sheet.paste(im,((j%2)*480,(j//2)*300))
    path=OUTPUT_DIR/f'scene_sheet_{k//4+1:02d}.jpg'
    sheet.save(path,quality=90)
    scene_sheets.append(path)
print('Scene contact sheets:',len(scene_sheets))
if scene_sheets:display(NotebookImage(filename=str(scene_sheets[0])))

# Construct a longitudinal crop strip for each canonical person.
person_review_imgs={}
for pid,tids in person_to_raw.items():
    options=sorted([(t,im) for tid in tids for t,im in tracks[tid]['crops']],key=lambda x:x[0])
    if not options:continue
    selected=[options[j] for j in sorted({0,len(options)//2,len(options)-1})]
    pic=PILImage.new('RGB',(900,400),'#171717');d=ImageDraw.Draw(pic)
    d.text((10,10),f'{pid}   source raw tracks: {tids}',fill='white')
    for j,(sec,im) in enumerate(selected):
        im=im.copy();im.thumbnail((280,340))
        pic.paste(im,(j*300+(300-im.width)//2,38+(340-im.height)//2))
        d.text((j*300+10,378),f'{sec:.1f}s',fill='white')
    path=OUTPUT_DIR/f'person_{pid}.jpg';pic.save(path,quality=92)
    person_review_imgs[pid]=path

## 8. Run Qwen3-VL locally on Colab GPU (no API)
**Scene pass:** examine before/after explosion and camera visibility. **Individual pass:** examine crops for visible cues such as being motionless on the ground, being carried, or receiving assistance. Crop-only analysis cannot establish a medical injury. Qwen may be wrong; outputs are proposals for review.

This cell uses 4-bit GPU quantization to fit a normal T4 more comfortably. For larger crowds, increase `MAX_PERSON_VLM_REVIEWS` and expect slower inference. If model loading runs out of memory, keep the default 2B model and reduce the number of scene sheets.

In [ ]:
scene_reviews=[]; person_reviews={};vlm_errors=[]

def parse_json_response(text):
    text=re.sub(r'^```(?:json)?\s*|\s*```$','',text.strip(),flags=re.IGNORECASE)
    try:return json.loads(text)
    except json.JSONDecodeError:
        start=text.find('{');end=text.rfind('}')
        if start>=0 and end>start:
            try:return json.loads(text[start:end+1])
            except json.JSONDecodeError:pass
        return {'parse_error':True,'unparsed_response':text}

if RUN_VLM:
    from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig
    vlm_processor=AutoProcessor.from_pretrained(VLM_MODEL,min_pixels=224*224,max_pixels=768*768)
    quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=torch.float16)
    vlm=Qwen3VLForConditionalGeneration.from_pretrained(
        VLM_MODEL,device_map='auto',dtype=torch.float16,
        quantization_config=quant,attn_implementation='sdpa').eval()
    def ask_vlm(image_file,prompt):
        with PILImage.open(image_file) as opened:
            img=opened.convert('RGB')
        msgs=[{'role':'user','content':[{'type':'image','image':img},
                                        {'type':'text','text':prompt}]}]
        batch=vlm_processor.apply_chat_template(msgs,add_generation_prompt=True,
                tokenize=True,return_dict=True,return_tensors='pt').to('cuda')
        with torch.inference_mode():
            result=vlm.generate(**batch,max_new_tokens=VLM_MAX_NEW_TOKENS,do_sample=False)
        raw=vlm_processor.batch_decode(result[:,batch['input_ids'].shape[-1]:],
                                      skip_special_tokens=True)[0]
        del batch,result
        return parse_json_response(raw)

    for i,path in enumerate(scene_sheets,1):
        prompt=('You are reviewing a timestamped surveillance video CONTACT SHEET (four chronological frames). '
                'Assess what is VISIBLE ONLY. Do not invent people or conclude anyone died or is injured. '
                'If smoke/dust/camera failure blocks the view, say so. The ID labels come from a separate tracker. '
                'Reply with ONLY valid JSON having keys: scene_summary (string), '
                'explosion_or_blast_visually_apparent (boolean), '
                'camera_view_obscured (boolean), visible_distress_evidence (string), '
                'uncertainties (string). Keep responses brief.')
        try:
            out=ask_vlm(path,prompt)
            out['sheet_number']=i
            scene_reviews.append(out)
        except Exception as e:
            vlm_errors.append(f'Scene sheet {i}: {type(e).__name__}: {e}')
        print('Scene sheet',i,'/',len(scene_sheets))

    # Prioritize posture flags and people near the event when the review cap is reached.
    def review_order(pid):
        ts=person_to_raw[pid]; s=[tracks[i] for i in ts]
        prior=int(any(x['posture_flag'] for x in s))
        before=int(any(x['saw_before'] for x in s))
        missing=int(before and not any(x['saw_after'] for x in s))
        return (-(prior*3+missing*2+before),min(x['first'] for x in s))
    ordered=sorted(person_to_raw,key=review_order)
    for j,pid in enumerate(ordered[:MAX_PERSON_VLM_REVIEWS],1):
        if pid not in person_review_imgs:continue
        prompt=(f'These are up to three timestamped crops of ONE TRACKED PERSON: {pid}. '
                'The same person is hypothesized across crops but identity matching can be mistaken. '
                'Only assess visibly obvious signs of needing assistance (for example, lying motionless on ground, '
                'being physically assisted or carried, or clear visible physical distress). '
                'Do NOT infer an injury from standing, walking, merely leaving the camera, smoke, or explosion alone. '
                'Do NOT diagnose injury or claim death. If unsure choose false and say uncertain. '
                'Reply as JSON only, with fields: visible_assistance_cue (boolean), evidence (string), '
                'confidence (one of low,medium,high), uncertainty (string).')
        try:person_reviews[pid]=ask_vlm(person_review_imgs[pid],prompt)
        except Exception as e:
            vlm_errors.append(f'{pid}: {type(e).__name__}: {e}')
        print('VLM reviewed',j,'/',min(len(ordered),MAX_PERSON_VLM_REVIEWS),pid)
    del vlm,vlm_processor
    torch.cuda.empty_cache();gc.collect()
else:
    print('RUN_VLM is False: no scene or person VLM review was performed.')

(OUTPUT_DIR/'vlm_scene_reviews.json').write_text(json.dumps(scene_reviews,indent=2,ensure_ascii=False))
(OUTPUT_DIR/'vlm_person_reviews.json').write_text(json.dumps(person_reviews,indent=2,ensure_ascii=False))
if vlm_errors:print('VLM errors:',vlm_errors[:4])
display(pd.DataFrame(scene_reviews).head())

## 9. Count each **canonical person** once and prepare the event report

The requested **people possibly needing help or checking** is the union of:

- **Visible assistance clues:** sustained horizontal posture or VLM-described visible evidence (neither proves injury).
- **Welfare-check priority:** visible shortly before the specified explosion but *not recognized afterward*. This remains **status unknown**, especially if the camera becomes obscured.

The union is deduplicated: if the same person has both reasons, they are counted **once**. If no explosion timestamp is set, the disappearance criterion is disabled. Output also separates people never reviewed by the VLM.

In [ ]:
rows=[]
for pid, tids in person_to_raw.items():
    chunks=[tracks[tid] for tid in tids]
    pre=any(t['saw_before'] for t in chunks)
    post=any(t['saw_after'] for t in chunks)
    unseen=bool(EVENT_TIME_SECONDS is not None and pre and not post)
    posture=any(t['posture_flag'] for t in chunks)
    vlm_r=person_reviews.get(pid,{})
    vlm_flag=vlm_r.get('visible_assistance_cue') is True and bool(vlm_r.get('evidence','').strip())
    manual_pos=any(t in MANUAL_VISIBLE_HELP_RAW_IDS for t in tids)
    manual_neg=any(t in MANUAL_DISMISS_HELP_RAW_IDS for t in tids)
    if manual_pos and manual_neg:raise ValueError(f'Conflicting manual review for {pid}')
    visual_flag=bool(manual_pos or ((posture or vlm_flag) and not manual_neg))
    welfare_check=bool(unseen)
    priority=bool(visual_flag or welfare_check)
    reasons=[]
    if posture and not manual_neg:reasons.append('sustained horizontal/sideways pose')
    if vlm_flag and not manual_neg:reasons.append('VLM visible cue: '+str(vlm_r.get('evidence',''))[:170])
    if unseen:reasons.append('seen before event, not reobserved afterward; status UNKNOWN')
    if manual_pos:reasons.append('manual confirmation of visible assistance cue')
    if manual_neg:reasons.append('manual dismissal of visible-cue flag')
    rows.append({'person_id':pid,'raw_tracker_ids':','.join(map(str,tids)),
                 'first_seen_s':round(min(x['first'] for x in chunks),2),
                 'last_seen_s':round(max(x['last'] for x in chunks),2),
                 'observations':sum(x['n'] for x in chunks),
                 'reconnected_from_multiple_tracks':len(tids)>1,
                 'seen_near_before_event':pre,'seen_after_event':post,
                 'unverified_after_event':unseen,
                 'pose_alert':posture,'vlm_reviewed':pid in person_reviews,
                 'vlm_visible_assistance_cue':vlm_flag,
                 'visible_assistance_cue':visual_flag,
                 'welfare_check_status_unknown':welfare_check,
                 'priority_check_union':priority,
                 'evidence_and_reason':'; '.join(reasons),
                 'vlm_uncertainty':str(vlm_r.get('uncertainty',''))})
people=pd.DataFrame(rows,columns=['person_id','raw_tracker_ids','first_seen_s','last_seen_s',
                                  'observations','reconnected_from_multiple_tracks',
                                  'seen_near_before_event','seen_after_event','unverified_after_event',
                                  'pose_alert','vlm_reviewed','vlm_visible_assistance_cue',
                                  'visible_assistance_cue','welfare_check_status_unknown',
                                  'priority_check_union','evidence_and_reason','vlm_uncertainty'])
people.to_csv(OUTPUT_DIR/'unique_people_and_help.csv',index=False)
counts={
    'estimated_unique_people':len(people),
    'people_with_possible_visible_assistance_cues':int(people.visible_assistance_cue.sum()),
    'people_unseen_after_event_status_unknown':int(people.unverified_after_event.sum()),
    'people_prioritized_for_assistance_or_welfare_check_DEDUPLICATED':int(people.priority_check_union.sum()),
    'people_returning_under_reconnected_track_ids':int(people.reconnected_from_multiple_tracks.sum()),
    'people_reviewed_by_vlm':int(people.vlm_reviewed.sum()),
    'people_not_reviewed_by_vlm':int((~people.vlm_reviewed).sum()),
    'confirmed_injuries': 'NOT DETERMINABLE FROM VIDEO'
}
print('=== RESULTS: ESTIMATES, NOT VERIFIED CASUALTIES ===')
for key,v in counts.items():print(key,':',v)
display(people)
report={'video':Path(VIDEO_PATH).name,'duration_analyzed_seconds':round(duration,2),
        'event_time_seconds':EVENT_TIME_SECONDS,'counts':counts,'scene_vlm_analysis':scene_reviews,
        'individuals':people.to_dict(orient='records'),'vlm_errors':vlm_errors,
        'limitations':['Tracker/ReID can split one person or merge visually similar people.',
                       'Not seen after an explosion is status unknown, NOT evidence of injury.',
                       'Camera occlusion is not individual disappearance; all unverified people may need a welfare check.',
                       'This is a research aid only: human verify identity and assistance flags.']}
(OUTPUT_DIR/'report.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
lines=['# Disaster video analysis (automatically generated)','',
       f"Video: {Path(VIDEO_PATH).name} | Analyzed: {duration:.1f} seconds",'']
lines += [f'- **{k.replace("_"," ")}**: {v}' for k,v in counts.items()]
lines+=['','## Scene analysis (Qwen3-VL)']
for r in scene_reviews:
    lines += [f"- Sheet {r.get('sheet_number','?')}: {r.get('scene_summary',r.get('unparsed_response','unable to parse'))}"]
lines += ['','## Per-person evidence']
for p in rows:
    lines += [f"- {p['person_id']} (tracks {p['raw_tracker_ids']}): {p['evidence_and_reason'] or 'No clear assistance evidence from analyzed frames'}"]
lines+=['','**Warning:** A lost view or unseen person is not proof of injury. All results require human review.']
(OUTPUT_DIR/'report.md').write_text('\n'.join(lines),encoding='utf-8')

## 10. Export a **final canonical-ID** annotated video and download everything
The final MP4 uses IDs such as `P001` **after de-duplication**. The CSV, JSON, contact sheets, VLM assessments, duplicate candidate table and MP4 are bundled into a ZIP. If you correct merges, rerun this cell to update the video.

In [ ]:
cap=cv2.VideoCapture(VIDEO_PATH)
output_mp4=OUTPUT_DIR/'canonical_people_annotated.mp4'
out=cv2.VideoWriter(str(output_mp4),cv2.VideoWriter_fourcc(*'mp4v'),fps,(W,H))
if not out.isOpened():raise RuntimeError('Cannot write final MP4')
try:
    for fr in frames:
        ok,img=cap.read()
        if not ok:break
        onscreen=set()
        for obs in fr['obs']:
            raw_id=obs['tid']
            if raw_id not in raw_to_person:continue
            pid=raw_to_person[raw_id]
            onscreen.add(pid)
            rr=people.loc[people.person_id==pid].iloc[0]
            bb=obs['box'];x1,y1,x2,y2=[int(x) for x in bb]
            col=(15,90,245) if rr.priority_check_union else (30,220,30)
            cv2.rectangle(img,(x1,y1),(x2,y2),col,2)
            cv2.putText(img,pid,(max(0,x1),max(20,y1-10)),cv2.FONT_HERSHEY_SIMPLEX,.65,col,2)
        caption=f'{fr["sec"]:.1f}s   visible: {len(onscreen)}   unique so far: (see CSV)'
        cv2.putText(img,caption,(10,35),cv2.FONT_HERSHEY_SIMPLEX,.65,(255,255,255),2)
        out.write(img)
finally:cap.release();out.release()

# Prefer H.264 when ffmpeg is available (for playback in notebooks).
playable=output_mp4
if shutil.which('ffmpeg'):
    converted=OUTPUT_DIR/'canonical_people_h264.mp4'
    cmd=['ffmpeg','-y','-loglevel','error','-i',str(output_mp4),'-vcodec','libx264',
         '-pix_fmt','yuv420p','-movflags','+faststart',str(converted)]
    p=subprocess.run(cmd,capture_output=True,text=True)
    if p.returncode==0:playable=converted

zip_path=shutil.make_archive('/content/disaster_ai_full_results','zip',OUTPUT_DIR)
print('REPORT:',OUTPUT_DIR/'report.md')
print('DEDUPLICATED COUNT:',counts['estimated_unique_people'])
print('POSSIBLE VISIBLE ASSISTANCE CUES:',counts['people_with_possible_visible_assistance_cues'])
print('UNVERIFIED AFTER EVENT:',counts['people_unseen_after_event_status_unknown'])
print('PEOPLE TO CHECK — COUNTED ONCE:',counts['people_prioritized_for_assistance_or_welfare_check_DEDUPLICATED'])
print('ZIP:',zip_path)
try:
    display(NotebookVideo(filename=str(playable),embed=False))
except Exception as exc:print('Video preview unavailable:',exc)
from google.colab import files
files.download(zip_path)

## Interpretation, accuracy, and references

- **This does not provide reliable casualty counts.** It highlights potential assistance cases and individuals unverified after an event.
- **A person unseen after a blast is not known injured.** If the camera is completely obscured, everyone may be unseen even if they are safe. Report them as *welfare-check candidates with unknown status*.
- **Double-counting is reduced, not eliminated.** A tracker can switch IDs; a ReID model can incorrectly merge two people wearing similar clothing. Verify the comparison sheet and adjust `FORCE_SAME` / `FORCE_DIFFERENT` when exact counts matter.
- **VLM outputs are observational suggestions.** A vision-language model can hallucinate and cannot diagnose unseen injuries. The final people count comes from the identity groups, **not** VLM guesses.
- **Event time is a user-provided observation.** It is not inferred from a scene description. Use `EVENT_TIME_SECONDS = None` if you cannot verify it.
- **Sampling limits:** only sampled frames and sampled crops are reviewed by the VLM. Short injury cues can be missed. Change limits for better coverage and always inspect the original video.
- **Data:** no inference API is called, but video data is uploaded into Colab and model weights are downloaded from Hugging Face/Ultralytics. Use appropriate privacy protections.
- **Licensing:** Qwen3-VL and DINOv2 model cards list Apache-2.0. Ultralytics software has AGPL/commercial licensing terms; check them before proprietary deployment.

Model documentation: [Ultralytics tracking](https://docs.ultralytics.com/modes/track/), [DINOv2-small](https://huggingface.co/facebook/dinov2-small), [Qwen3-VL-2B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct).